# Interactive exploration

Scratchpad on top of the `libs2026` library. Run `scripts/01_prepare_data.py` first.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np

from libs2026 import Config, Preprocessor, build_features, load_shots, load_index

cfg = Config.load('../configs/default.yaml')
index = load_index(cfg)
index.head()

In [ ]:
# Raw versus preprocessed spectrum of a single sample
pre = Preprocessor.from_config(cfg)
shots = np.asarray(load_shots(cfg, 'train_001', mmap=False))
processed = pre(shots)
print(shots.shape, '->', processed.shape)

In [ ]:
features = build_features(cfg, pre, n_jobs=8)
train = features.subset('train')
features, train.X.shape

In [ ]:
# Where do the classes differ? Fisher-style ratio per wavelength.
classes = np.unique(train.y)
means = np.stack([train.X[train.y == c].mean(axis=0) for c in classes])
within = np.stack([train.X[train.y == c].std(axis=0) for c in classes]).mean(axis=0)
score = means.std(axis=0) / np.where(within > 0, within, 1)

top = np.argsort(score)[-25:][::-1]
for i in top[:10]:
    print(f'{features.wavelength[i]:8.3f} nm   F = {score[i]:.2f}')